# 01 · export v3 — database → canonical parquet (regeneration)

**Kernel: `fttl-v3` (env-v3, Python 3.11, xgboost 3.2.0).** v3 saved nothing but its two pickles,
so this notebook re-runs the repo's own chain — DB extract → enrichment joins → cc-rule →
stateless stage → `p146_pipeline` transform — and writes what training never persisted.
Company network + DB credentials required.

⚠️ Two caveats carried into every artefact written here:
- **Enrichment is as-of-now, not as-of-training** — hpi/thatcham joins use today's tables, so the
  regenerated matrix can differ from what training saw (README § enrichment).
- The import cell uses the module names **as transcribed** from the repo (2026-08-08). If they
  live under a package prefix, adjust the imports — do not guess further names.


In [ ]:
import sys
from pathlib import Path

import joblib

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import config                      # noqa: E402

import xgboost                     # noqa: E402
assert xgboost.__version__ == config.xgboost_pin("v3")

# repo modules, names as transcribed — adjust the prefix if the repo packages them
import train_helper                # noqa: E402
import pipelines                   # noqa: E402
import project_params              # noqa: E402

KEY = project_params.KEY           # the claim id — copy this string into config columns.claim_id
print("claim id column:", KEY)


In [ ]:
# DB extract + target + maturation + enrichments — the repo's own load chain
data = train_helper.load_data()                       # SELECT * FROM …p146_extract_v3 (+ Fttl)
mature = train_helper.subset_by_date(data)
enriched = train_helper.merge_in_enrichments(mature, train_helper.load_enrichement_data())
# cc-rule flags: follow the train script here if add_cc_rule is not reachable via train_helper
print(enriched.shape)


In [ ]:
prep = joblib.load(config.path("preprocessor", "v3", "real"))   # p146_pipeline.pkl
stateless = pipelines.stateless_pipeline(enriched)
proc = prep.transform(stateless)

est = joblib.load(config.path("model", "v3", "real"))           # p146_model.pkl
feats = list(est.get_booster().feature_names)
scores_np = est.predict_proba(proc.select(feats).to_pandas())[:, 1]


In [ ]:
import pandas as pd

DATE, OBSERVED = config.column("v3", "date"), config.column("v3", "observed")
proc_pd = proc.to_pandas() if hasattr(proc, "to_pandas") else proc
raw_pd  = enriched.to_pandas() if hasattr(enriched, "to_pandas") else enriched

def write(df, kind):
    p = config.path(kind, "v3", "real")     # all four resolve to the fallback tree for v3
    p.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(p, index=False)
    print(kind, "->", p, len(df), "rows")

write(raw_pd.rename(columns={KEY: "claim_id"}), "raw_dataset")   # freeze THIS run's extract
write(proc_pd.rename(columns={KEY: "claim_id"}), "processed_inputs")
write(proc_pd[[KEY, DATE, OBSERVED]].rename(
    columns={KEY: "claim_id", DATE: "date", OBSERVED: "observed"}), "targets")
write(pd.DataFrame({"claim_id": proc_pd[KEY].values, "model_v3_score": scores_np}), "scores")
